In [1]:
import numpy as np
import k3d

In [2]:
class Detector:
    def __init__(self, R: float = 2.0,
                       N: int = 40,
                       r: float = 0.1,
                       a: float = 0.5):
        """Cферический детектор"""
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.p_positions = self.generate_positions()

    def generate_positions(self) -> np.ndarray:
        """Равномерное размещениена сфере"""
        i = np.arange(self.N)

        z = self.R * (1 - 2 * (i + 0.5) / self.N)

        phi = i * np.pi * (3 - np.sqrt(5))

        # Радиус в плоскости xy
        rho = np.sqrt(self.R**2 - z**2)

        x = rho * np.cos(phi)
        y = rho * np.sin(phi)

        return np.column_stack((x, y, z))

    def is_on_pmt(self, point) -> bool:
        """Проверка попадания точки """

        point = np.asarray(point, dtype=float)
        if point.shape != (3,):
            raise ValueError("Точка должна иметь 3 координаты")

        if not np.isclose(np.linalg.norm(point),
                          self.R, atol=1e-8, rtol=0):
            return False

        # Расстояния до центров всех ФЭУ
        distances = np.linalg.norm(
            self.p_positions - point, axis=1
        )

        # Попадание хотя бы в один фотоумножитель
        return bool(np.any(distances <= self.r + 1e-10))

In [3]:
detector = Detector()

In [4]:
# Все центры ФЭУ должны лежать на сфере
radii = np.linalg.norm(detector.p_positions, axis=1)
assert np.allclose(radii, detector.R)

In [ ]:
print("Координаты первых 5 ФЭУ: \n", detector.p_positions[:5])

Координаты первых 5 ФЭУ: 
 [[ 0.44440972  0.          1.95      ]
 [-0.56035183  0.51332818  1.85      ]
 [ 0.08464959 -0.96453846  1.75      ]
 [ 0.68769743  0.89697951  1.65      ]
 [-1.2446015  -0.22015248  1.55      ]]


In [6]:
plot = k3d.plot()

# Поверхность сферы
theta = np.linspace(0, np.pi, 40)
phi = np.linspace(0, 2 * np.pi, 80)

theta_grid, phi_grid = np.meshgrid(theta, phi)

x = detector.R * np.sin(theta_grid) * np.cos(phi_grid)
y = detector.R * np.sin(theta_grid) * np.sin(phi_grid)
z = detector.R * np.cos(theta_grid)


In [9]:
plot += k3d.surface(
    z.astype(np.float32),
    bounds=[
        -detector.R, detector.R,
        -detector.R, detector.R
    ],
    color=0x87CEEB,
    opacity=0.15
)

plot += k3d.points(
    detector.p_positions.astype(np.float32),
    point_size=0.2,
    color=0xFF4500
)

plot.display()

Output()